# Validate Customers
Validates raw customers table for further transformation.

In [0]:
%sql
SELECT * 
FROM ecommerce_catalog.bronze.customers_raw
ORDER BY customer_id;

In [0]:
SELECT
  count(*) AS total_rows,
  count(DISTINCT customer_id) AS unique_customer_id,
  SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS missing_customers,
  SUM(CASE WHEN first_name IS NULL THEN 1 ELSE 0 END) AS missing_first_name,
  SUM(CASE WHEN last_name IS NULL THEN 1 ELSE 0 END) AS missing_last_name,
  SUM(CASE WHEN email IS NULL THEN 1 ELSE 0 END) AS missing_emails,
  SUM(CASE WHEN registration_date IS NULL THEN 1 ELSE 0 END) AS missing_registration_dates,
  SUM(CASE WHEN country IS NULL THEN 1 ELSE 0 END) AS missing_countrys,
  SUM(CASE WHEN city IS NULL THEN 1 ELSE 0 END) AS missing_cities,
  SUM(CASE WHEN process_date IS NULL THEN 1 ELSE 0 END) AS missing_process_dates,
  SUM(CASE WHEN source_file IS NULL THEN 1 ELSE 0 END) AS missing_source_files
FROM ecommerce_catalog.bronze.customers_raw;

In [0]:
SELECT customer_id, count(*) AS row_count
FROM ecommerce_catalog.bronze.customers_raw
GROUP BY customer_id
HAVING count(*) > 1;

In [0]:
SELECT *
FROM ecommerce_catalog.bronze.customers_raw
WHERE customer_id = "C0008" OR customer_id = "C0181";

In [0]:
SELECT email
FROM ecommerce_catalog.bronze.customers_raw
WHERE email LIKE "%@";

In [0]:
SELECT 
    SUM(CASE WHEN customer_status = "Active" THEN 1 ELSE 0 END) AS status_Active,
    SUM(CASE WHEN customer_status = "active" THEN 1 ELSE 0 END) AS status_active,
    SUM(CASE WHEN customer_status = "ACTIVE" THEN 1 ELSE 0 END) AS status_ACTIVE,
    SUM(CASE WHEN customer_status = "inactive" THEN 1 ELSE 0 END) AS status_inactive,
    SUM(CASE WHEN customer_status = "Inactive" THEN 1 ELSE 0 END) AS status_Inactive,
    SUM(CASE WHEN customer_status = "INACTIVE" THEN 1 ELSE 0 END) AS status_INACTIVE
FROM ecommerce_catalog.bronze.customers_raw


### **Observations:**
- Two duplicate `customer_id` entries (`C0008` and `C018`) were identified. Investigation revealed that both cases are complete duplicates with identical attributes.
- Two email addresses lack a domain and are therefore invalid email formats.
- The `registration_date` column contains multiple different date formats and must be unified.
- Inconsistent casing (e.g., uppercase and lowercase) was identified in the `customer_status` column.
- No missing (`NULL`) values were identified.

### **Silver Layer Actions:**
- Remove complete duplicates.
- Standardize the date format of `registration_date`.
- Standardize casing in `customer_status`.
- Document invalid email addresses without applying automatic corrections, as the missing domains cannot be unambiguously reconstructed.